In [2]:
# Following example from https://pub.towardsai.net/efficient-camera-stream-with-python-b6adf93fab32

import platform
import subprocess
import time
import cv2
import numpy as np

# Video Stream Class
class VideoStreamCV:
    def __init__(self, src: int, fps: int, resolution: tuple[int, int]):
        self.src = src
        self.fps = fps
        self.resolution = resolution
        self.cap = self._open_camera()
        self.wait_for_cam()
        
    def _open_camera(self):
        cap = cv2.VideoCapture(self.src)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, self.resolution[0])
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, self.resolution[1])
        fourcc = cv2.VideoWriter_fourcc(*"MJPG")
        cap.set(cv2.CAP_PROP_FOURCC, fourcc)
        cap.set(cv2.CAP_PROP_FPS, self.fps)
        return cap
    
    def read(self):
        ret, frame = self.cap.read()
        if not ret:
            return None
        return frame
    
    def release(self):
        self.cap.release()
        
    def wait_for_cam(self):
        for _ in range(30):
            frame = self.read()
        if frame is not None:
            return True
        return False

In [ ]:
# Following example for testing read time from https://pub.towardsai.net/efficient-camera-stream-with-python-b6adf93fab32

def timeit(func):
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        t1 = time.perf_counter()
        print(f"Main function time: {round(t1-t0, 4)}s")
        return result
    
    return wrapper

def computation_task():
    for _ in range(5000000):
        9999 * 9999

@timeit
def run(cam: VideoStreamCV, run_task: bool):
    timer = []
    for _ in range(100):
        t0 = time.perf_counter()
        cam.read()
        timer.append(time.perf_counter() - t0)

        if run_task:
            computation_task()

    cam.release()
    return round(np.mean(timer), 4)


fps = 30
resolution = (300, 300)

for run_task in [False, True]:
    cam = VideoStreamCV(src=0, fps=fps, resolution=resolution)
    print(f"CV2, task {run_task}:")
    print(f"Mean frame read time: {run(cam=cam, run_task=run_task)}s\n")

CV2, task False:
Main function time: 3.5164s
Mean frame read time: 0.0336s

CV2, task True:
Main function time: 8.9903s
Mean frame read time: 0.0199s



In [12]:
# Displaying video capture for testing
# First, lets see how stack does it:

# Source - https://stackoverflow.com/a/11449901
# Posted by sastanin, modified by community. See post 'Timeline' for change history
# Retrieved 2026-11-08, License - CC BY-SA 3.0

# cv2.namedWindow("preview")
# vc = cv2.VideoCapture(0)

# if vc.isOpened(): # try to get the first frame
#     rval, frame = vc.read()
# else:
#     rval = False

# while rval:
#     cv2.imshow("preview", frame)
#     rval, frame = vc.read()
#     key = cv2.waitKey(20)
#     if key == 27: # exit on ESC
#         break

# cv2.destroyWindow("preview")
# vc.release()


# Now, lets do it using our architecture:
def display_camera_feed():
    cam = VideoStreamCV(src=0, fps=fps, resolution=resolution)
    if cam.cap.isOpened():
        rval, frame = cam.cap.read()
    else:
        rval = False
        
    while rval:
        cv2.imshow("preview", frame)
        rval, frame = cam.cap.read()
        key = cv2.waitKey(20)
        if key == 27: #exit on ESC
            break
        
    cv2.destroyWindow("preview")
    cam.cap.release()
    return None

display_camera_feed()


In [19]:
# Now, we need to recognize hand shapes so that we can apply the trained CNN
# Following guidance of https://www.youtube.com/watch?v=RRBXVu5UE-U
# and https://github.com/google-ai-edge/mediapipe/issues/6204

#!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = "hand_landmarker.task"
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),        # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),        # Index
    (0, 9), (9, 10), (10, 11), (11, 12),   # Middle
    (0, 13), (13, 14), (14, 15), (15, 16), # Ring
    (0, 17), (17, 18), (18, 19), (19, 20), # Pinky
    (5, 9), (9, 13), (13, 17)              # Palm connections
]

def to_pixel(x_norm: float, y_norm: float, w: int, h: int) -> tuple[int, int]:
    # keep to [0,1] to avoid occasional out-of-range artifacts
    x = min(max(x_norm, 0.0), 1.0)
    y = min(max(y_norm, 0.0), 1.0)
    return int(x * w), int(y * h)

def draw_hand_landmarks_tasks_only(
    image_bgr: np.ndarray,
    hand_landmarks_list,
    connections=HAND_CONNECTIONS,
    draw_points=True,
    draw_connections=True,
    point_radius=10,
    point_thickness=-1,
    line_thickness=7,
):
    annotated = image_bgr.copy()
    h, w = annotated.shape[:2]

    for hand_landmarks in hand_landmarks_list:
        # Convert normalized landmarks to pixel coords
        pts = [to_pixel(lm.x, lm.y, w, h) for lm in hand_landmarks]

        if draw_connections:
            for a, b in connections:
                cv2.line(annotated, pts[a], pts[b], (255, 255, 255), line_thickness)

        if draw_points:
            for (x, y) in pts:
                cv2.circle(annotated, (x, y), point_radius, (0, 0, 255), point_thickness)

    return annotated


def hand_tracking():
    cam = VideoStreamCV(src=0, fps=fps, resolution=resolution)
    if cam.cap.isOpened():
        rval, frame = cam.cap.read()
    else:
        rval = False
    
    i = 0
    while rval:
        base_options = python.BaseOptions(model_asset_path=model_path)
        options = vision.HandLandmarkerOptions(
            base_options=base_options,
            num_hands=2,
            min_hand_detection_confidence=0.5,
            running_mode=vision.RunningMode.IMAGE
        )
        
        image = cv2.imwrite('recorded_frames/{index}.png'.format(index=i), frame)

        with vision.HandLandmarker.create_from_options(options) as landmarker:
            cv_image = cv2.imread('recorded_frames/{index}.png'.format(index=i))
            if cv_image is None:
                print("Image not found:", image)
                return
    
            rgb_image = cv2.cvtColor(cv_image, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
    
            result = landmarker.detect(mp_image)
    
        if result.hand_landmarks:
            annotated = draw_hand_landmarks_tasks_only(cv_image, result.hand_landmarks)
            cv2.imshow("capture image", annotated)
        else:
            cv2.imshow("capture image", cv_image)
                
        rval, frame = cam.cap.read()
        i += 1
        key = cv2.waitKey(20)
        if key == 27: #exit on ESC
            break
        
    cv2.destroyWindow("capture image")
    cam.cap.release()
    return None

hand_tracking()


W0000 00:00:1794183490.410617 19054462 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794183490.416737 19054462 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794183490.798297 19054521 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794183490.802435 19054521 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794183491.164175 19054546 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794183491.169561 19054546 inference_feedback_manager.cc:121] Feedback ma